todo: add docs for inital seelig v1 orth creation

In [1]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster
from pathlib import Path

%load_ext autoreload
%autoreload 2

2026-02-12 16:57:15.425582: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-12 16:57:15.429221: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/2024a/software/code-server/4.103.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cUR

In [2]:
local=True
if local:
    cluster=LocalCluster(
        n_workers=4,
        threads_per_worker=2
    )
    client=Client(cluster)
else:
    cluster=SLURMCluster(
        cores=3,#cores per slurm job
        memory="512G",#memory per slurm job
        processes=3,#dask workers per slurm job,
        job_extra_directives=["-p week", 
            f"--job-name=simclust_worker",
            "--resources \"FIT=1\"",
            f"--time=72:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=2)
    #cluster.scale(jobs=20)
    #cluster.scale(jobs=20)
    client = Client(cluster,
            timeout=f"{5*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s"  # Worker heartbeat interval
        )

In [3]:
data_root=Path("/nfs/roberts/project/pi_skr2/shared/tabula_data")
client.dashboard_link

'http://127.0.0.1:8787/status'

In [4]:
primordial=scm.ortho.load(client,data_root/"seelig","ortho_seelig_v1.5")

In [5]:
primordial.extract_params(client)

In [6]:
primordial.save(data_root/"seelig","ortho_seelig_v2")

2026-02-12 17:02:11,636 - distributed.worker - ERROR - Compute Failed
Key:       _extract_mu-edc466240afa555fc05fa4c65da42e81
State:     executing
Task:  <Task '_extract_mu-edc466240afa555fc05fa4c65da42e81' _extract_mu(, ...)>
Exception: "ValueError('No index can be less than zero')"
Traceback: '  File "/nfs/roberts/project/pi_skr2/mcn26/tabula-rasa/notebooks/object_creation/orthos/scMPRAforge/core.py", line 2180, in _extract_mu\n    row_labeling=undo_one_hot_encoding(X)\n  File "/nfs/roberts/project/pi_skr2/mcn26/tabula-rasa/notebooks/object_creation/orthos/scMPRAforge/utils.py", line 146, in undo_one_hot_encoding\n    row_sums = subdf.sum(axis=1)\n  File "/home/mcn26/.conda/envs/env_tzinb/lib/python3.10/site-packages/pandas/core/frame.py", line 11697, in sum\n    result = super().sum(axis, skipna, numeric_only, min_count, **kwargs)\n  File "/home/mcn26/.conda/envs/env_tzinb/lib/python3.10/site-packages/pandas/core/generic.py", line 12571, in sum\n    return self._min_count_stat_funct

ValueError: No index can be less than zero

In [31]:
#print(type(z))
#print(z.dtypes.head())

for c in z.columns:
    arr = z[c].array
    print(c, type(arr))

Intercept <class 'pandas.core.arrays.sparse.array.SparseArray'>
C(cre_id, contr.treatment(base='reference'))[T.AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATATTTGGAGGTATCTGCCAAGCCCAGTCTCTCTGCCGCAGACAATTCCTGTAACCACACACTTCCTCTGCCAAGAGGGTGGAACCAAGGTCATACTCCCTC] <class 'pandas.core.arrays.sparse.array.SparseArray'>
C(cre_id, contr.treatment(base='reference'))[T.AAAATTAAACACTCGGGACTTGTCCGGGCATGCTGGCTGACTTGGCTGAACTTTGGCTGTTGGTATGGTGACGTGACATAGCTTTGCAACGTACTGTCTGTAACCTTGGACTTTGCACAAACGTACAAAGCATGCCGGAGGGGAA] <class 'pandas.core.arrays.sparse.array.SparseArray'>
C(cre_id, contr.treatment(base='reference'))[T.AAAATTAGCCGGGCGTGGTAGCAGGCGCCTGTAGTCCCAGCTACTCTGGAGGCTGAGGCAGGAAAATGGCGGGAACCCGAGAGGCGGAGCTTGCAGTGAGCCGATATCGCGCCACTGAACTCCAGCCCGGACAACAGAGCGAGAC] <class 'pandas.core.arrays.sparse.array.SparseArray'>
C(cre_id, contr.treatment(base='reference'))[T.AAACAGGTCGGGGGTTAATCCATACACACGCTGGGGTTTTGCCCAGGCAGGCCGGAATGGTCAACCTTTGGTCTTTGTACCGTCATGTTGACCTCGTCTGGACGGTTGAACTTTGCCCGTGTGCATTGGTACACTCGGTATGTAC

In [35]:
n_rows, n_cols = z.shape
print(n_rows, n_cols, n_rows * n_cols)

6041409 1330 8035073970


In [34]:
bad = []
for c in z.columns:  # limit if huge
    arr = z[c].array
    if hasattr(arr, "sp_index"):
        idx = arr.sp_index.indices
        if len(idx) and idx.min() < 0:
            bad.append((c, int(idx.min())))
bad[:10], len(bad)

([], 0)

In [12]:
primordial.by_cell_type_design["reference"].result()["model_type"]

'contrastable'

In [26]:
primordial.by_cre_design["reference"].result()["nb_regressors"].index

Index([ 330718,  330719,  330720,  330721,  330722,  330723,  330724,  330725,
        330726,  330727,
       ...
       3858364, 3858365, 3858366, 3858367, 3858368, 3858369, 3858370, 3858371,
       3858372, 3858373],
      dtype='int64', length=19454)

In [10]:
z=primordial.by_cell_type_design["reference"].result()["nb_regressors"]

In [14]:
z

,Intercept,"C(cre_id, contr.treatment(base='reference'))[T.AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATATTTGGAGGTATCTGCCAAGCCCAGTCTCTCTGCCGCAGACAATTCCTGTAACCACACACTTCCTCTGCCAAGAGGGTGGAACCAAGGTCATACTCCCTC]","C(cre_id, contr.treatment(base='reference'))[T.AAAATTAAACACTCGGGACTTGTCCGGGCATGCTGGCTGACTTGGCTGAACTTTGGCTGTTGGTATGGTGACGTGACATAGCTTTGCAACGTACTGTCTGTAACCTTGGACTTTGCACAAACGTACAAAGCATGCCGGAGGGGAA]","C(cre_id, contr.treatment(base='reference'))[T.AAAATTAGCCGGGCGTGGTAGCAGGCGCCTGTAGTCCCAGCTACTCTGGAGGCTGAGGCAGGAAAATGGCGGGAACCCGAGAGGCGGAGCTTGCAGTGAGCCGATATCGCGCCACTGAACTCCAGCCCGGACAACAGAGCGAGAC]","C(cre_id, contr.treatment(base='reference'))[T.AAACAGGTCGGGGGTTAATCCATACACACGCTGGGGTTTTGCCCAGGCAGGCCGGAATGGTCAACCTTTGGTCTTTGTACCGTCATGTTGACCTCGTCTGGACGGTTGAACTTTGCCCGTGTGCATTGGTACACTCGGTATGTAC]","C(cre_id, contr.treatment(base='reference'))[T.AAACATTCATGTCAGGGCATGTGGGCTTGTAACTTTGAACCCCTGCGCGATCAAACAAAGGTTGAGCGAAAATCCACCCGTGCAAACATGTCCGGGCATGCCTGCTGGCGAACGTGCGAACTCCGACCGGAGAGTTTGGCGCACG]","C(cre_id, contr.treatment(base='reference'))[T.AAACATTCCGGGGGGTAGACATGTCCGGGCATGTTTGCCGGCCGGCAGCAAAGTACAAACATTGCATAATGCCGGGGTTTGTACATGCCCGGGGGCAAACATGCCCGGGCATGTTCAAAGTTCGCAATGTTCAAAGGGACCTTTG]","C(cre_id, contr.treatment(base='reference'))[T.AAACCAAATGACTCAACACCCACATCCAACAAAAAACAAATGACTCACAACAACGATAACTCAAAAGAAAGCGCAGTCGCGGAAGCGCCGCCCACATCCAAACCAAAAAACCAAAAAACCACAACCAATCCAACCAAAGATAGAG]","C(cre_id, contr.treatment(base='reference'))[T.AAACCATATCTTAGTAGCTTATTATTACGACAGAAAACAGAACCGGCGCGTCACTTCCGGGTTCCGCGAATTACATTTTCTAGCAAACGGAGACGCGTCGTCGATAAGCCGATGCTTATTCGTAAATTGCAGTCTCCCTAGCGGA]","C(cre_id, contr.treatment(base='reference'))[T.AAACCGTCCGGCCGGTGGTTTATGTAATGGTTTATGTAACCGGTGATCTAACCGGTGGTCCAATGGTTTGTCCGTTGATCTAAGCGGTAATGTAAGCGTTTGTGCAATGGTTAATGCAACGGGTTATGTAAGCGGTGATGTAATG]",...,"C(cre_id, contr.treatment(base='reference'))[T.TTTGTAGCCTCCGACTTACCGTAAAATGTGGGCAACAAACACTACCGGGCATGCCTGGGGAGTGCCCATTCTGAGACATGTTTGGTAACAACGCCCAGTTTGTTTAACCACAGAAACGCTCGCGAGTCCAAAATTGTCGACCTGA]","C(cre_id, contr.treatment(base='reference'))[T.TTTGTGCAGAGTGAATTTGAGCTGCTGTGTTGGGAGCATGTCTGAGGTCAAAGGGCACAGTCCCACACCCGAAAGTTGCAAGGCCAAGGCTGTACTCATCAAGCTATGTATGGGGAACAACCCACTCTCCTGTATGACAGCTCTT]","C(cre_id, contr.treatment(base='reference'))[T.TTTGTTGAGCTGCTTCTACCTTTTCCGGGAAGTTGCAGCGGGTTTCGCGAGATAAGGAAGTGACTTATCTCATCCTCATTGGGCGATGTCAGTTCAGAGTAGTAACGCGCCTCCCCCCCATATAAGGACGCGCCAATTCGCGGAT]","C(cre_id, contr.treatment(base='reference'))[T.TTTTAACTGTCACCAAGGGCTGGGATAGCAAATATTGCATCAACTCTGCAAGGTGTTTCCTGGGTTTTGTGGACAGGGGGGTCGGGGGAGGTGCAATCTCCATTCTTGTGAAGGCCAGATAAAGACTTCTGCTGCAAGCCACTAT]","C(cre_id, contr.treatment(base='reference'))[T.TTTTATTGGTTTATCGGCCTATTGGCTTATTGGCCTATGCGCCCTTTTTTTTATTGGCGTATCGGCCTATTGGCTTATCCGCGCCCCCTTTTATCCCCCCCCTTTTCTATTGGCGCTTCCCTTTTTGCCTTCCCGTTTTTATCTT]","C(cre_id, contr.treatment(base='reference'))[T.TTTTTCTTACGCGGGTCATCACTCGTATGAAATGACTCACGCGACTTCTGATGAGTAATCACGCAATCGGTTCCGCGAAGTAAGATCTGCTATCGGACGCGGCGCTTCCTTCTTATATTGGGAAATCGCTTCATTAGTATGAGGC]","C(cre_id, contr.treatment(base='reference'))[T.TTTTTGTTGGCGCGCGCGCCTGAAGCGGGACTGCCAGGTGGCGCGCGCGCCTGAAGGCGATTATGGCGCGCGCGCCTGACACGGGGTATTGTCCTGGAGTTATGCGGCTACAGGAATGGGACGGCGCCACTGCGGGGTTTGCCGG]","C(cre_id, contr.treatment(base='reference'))[T.TTTTTGTTTGACCCCTGTAATGTTTGTTCCCAGGGAACATGCCGGGGCACGTGACCTCTGTCCGGTAATGTTTGAACAAAGGTCACATGCCTGGGCATGTCCTTTGAACAAAGCGAACATTACATAAACGTTCACAGGTCACCTG]","C(cre_id, contr.treatment(base='reference'))[T.TTTTTGTTTGTACAAAGCTCTGTTTGACCCCTGCGGGCATGCCGGGGCACGTGACCTCTGTCCGGTAATGTTTGAACAAAGCGAACATTACATGAACCTCTGCTGCTCTCTCTGTTTGAACATTCAAAGCGAACATTACATAAAC]","C(cre_id, contr.treatment(base='reference'))[T.TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTTACAGGTCTACATTACGTAGTCTGACATTTTTGGTTTTTTGTTTTTTTTGTTTTGTGCCGGCATGCCAGGGGTCATGTTCGGACATGTGACCGAGGTTTT]"
0,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...

In [24]:
z.index

Index([       0,        2,        4,        6,        8,        9,       15,
             17,       19,       21,
       ...
       12671646, 12671649, 12671651, 12671652, 12671659, 12671660, 12671661,
       12671663, 12671667, 12671669],
      dtype='int64', length=6041409)

In [19]:
z.sum(axis=1)

ValueError: No index can be less than zero

In [18]:
scm.undo_one_hot_encoding(z)

ValueError: No index can be less than zero

In [ ]:
client.close()
cluster.close()